# LLM 推理为什么慢：Prefill、Decode 与 KV Cache

> 上一章解决了“logits 怎么选成 Token”。这一章往下走一层：**为了得到 logits，模型到底做了哪些计算？**
>
> 这是后面所有推理优化的地基。读懂 Prefill、Decode、KV Cache、MQA/GQA、memory-bound 之后，再看 vLLM / SGLang / TensorRT-LLM 的技术报告就不会只剩名词。

一个请求可以拆成两段：

```text
Prompt tokens ── Prefill ──> 第一个 Token
                           ↓
                      Decode step
                           ↓
                      Decode step
                           ↓
                          ...
```


## 1. Prefill 和 Decode 是两种完全不同的工作负载

- **Prefill**：一次处理完整 Prompt，矩阵较大，并行度高，通常更偏 compute-bound。
- **Decode**：每一步只新增一个 Token，却要读取整套权重和越来越长的 KV Cache，通常更偏 memory-bound。

所以 TTFT（首 Token 延迟）和 TPOT（后续 Token 间隔）不能混成一个“延迟”指标。


In [ ]:
prompt_tokens = 2000
output_tokens = 300

print("Prefill: 一次处理", prompt_tokens, "个 prompt token")
print("Decode : 之后执行", output_tokens, "次串行 step")
print("结论：长 prompt 主要影响 TTFT；长输出主要影响 TPOT 和总时延。")


## 2. 没有 KV Cache，会重复算什么？

自回归生成每一步都把新 Token 接到历史后面。

```text
Step 1: [A]             -> token1
Step 2: [A, token1]     -> token2
Step 3: [A, token1, t2] -> token3
```

如果每一步都重新计算历史 Token 的 K/V，历史前缀会被反复处理。

下面的 `1+2+...+N` 只是一个**重复处理历史 token 数量的代理量**，不是完整 Transformer FLOPs complexity。


In [ ]:
def repeated_prefix_proxy(n):
    return n * (n + 1) // 2

for n in [10, 100, 1000]:
    naive = repeated_prefix_proxy(n)
    cached = n
    print(f"N={n:4d}  naive proxy={naive:8d}  cache-new-token proxy={cached:5d}")


## 3. KV Cache：把历史 K/V 留下来

Attention 在 Decode 时真正需要的是：

```text
new Query
   ×
historical K/V
```

历史 K/V 可以缓存，不必重新从历史 hidden states 计算。

但 KV Cache 不是“让 Attention 变成 O(N)”。新 Query 仍要和越来越长的历史 KV 做注意力；它省掉的是**历史 K/V 的重复计算**。


## 4. KV Cache 为什么会变成新的显存大户？

一个粗略公式：

$$
\text{KV bytes}
\approx
2 \times L \times T \times H_{kv} \times D \times B \times \text{bytes}
$$

其中 `2` 表示 K 和 V，`L` 是层数，`T` 是上下文长度，`H_kv` 是 KV head 数。


In [ ]:
def kv_cache_gb(layers, tokens, kv_heads, head_dim, batch, bytes_per_elem=2):
    total = 2 * layers * tokens * kv_heads * head_dim * batch * bytes_per_elem
    return total / 1e9

cfgs = [("MHA 32 KV heads", 32), ("GQA 8 KV heads", 8), ("MQA 1 KV head", 1)]
for name, kvh in cfgs:
    size = kv_cache_gb(32, 8192, kvh, 128, batch=16)
    print(f"{name:<18}: {size:6.2f} GB")


## 5. MHA、GQA、MQA：为什么招聘 JD 经常一起出现？

它们本质都在回答：**K/V 要保留多少份？**

```text
MHA: 每个 Query head 都有自己的 K/V
GQA: 一组 Query heads 共享一组 K/V
MQA: 所有 Query heads 共享一组 K/V
```

GQA / MQA 的直接收益之一，就是减少 KV Cache 和 Decode 阶段的内存带宽压力。


## 6. 为什么 Decode 经常是 memory-bound？

以 7B BF16 权重为例，仅权重约 14 GB。单 token Decode 的矩阵很“瘦”，GPU 算力未必吃得满，但每一步都要搬大量权重。

这也是为什么后面会出现三条不同路线：

```text
权重太大 / 搬得太慢
    -> Quantization

一次只能确认一个 token
    -> Speculative Decoding

多请求时 KV Cache / 调度太乱
    -> PagedAttention / Continuous Batching / PD 分离
```


In [ ]:
params = 7e9
weight_bytes_bf16 = params * 2
weight_bytes_int4_ideal = params * 0.5

print("7B BF16 理论权重:", weight_bytes_bf16/1e9, "GB")
print("7B INT4 理论权重:", weight_bytes_int4_ideal/1e9, "GB")
print("下一章：为什么少几个 bit 能明显改变 Decode 的数据搬运。")


## 小结：看到这些词，先放回正确位置

| 名词 | 它主要解决什么 |
|---|---|
| Prefill | 一次处理 Prompt |
| Decode | 串行生成后续 Token |
| TTFT | Prefill + 排队等带来的首 Token 等待 |
| TPOT / ITL | Decode 阶段的 Token 间隔 |
| KV Cache | 避免历史 K/V 重算 |
| GQA / MQA | 减少 KV Cache / 带宽 |
| memory-bound | 数据搬运限制速度，而不是算力 |

下一章只解决一个问题：

> **模型权重太大、每步搬运太贵，怎么办？**
